# 시험 환경 점검

시험 볼 컴퓨터에서 **Kernel → Restart & Run All**. 모든 셀이 ✅로 끝나면 준비 완료.

- 데이터셋을 미리 받아 둡니다 (시험 중 다운로드 지연 방지)
- CPU 속도를 재서 시험 때 epoch를 얼마나 줄 수 있는지 알려줍니다

In [ ]:
import sys, importlib
print("Python", sys.version.split()[0])
for name in ["tensorflow", "numpy", "pandas", "matplotlib", "sklearn", "scipy", "seaborn", "skimage", "PIL"]:
    try:
        m = importlib.import_module(name)
        print("✅", name, getattr(m, "__version__", ""))
    except Exception as e:
        print("❌", name, "→ 설치 필요:", e)

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
print("✅ TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU") or "없음 (CPU 사용)")

## 데이터셋 미리 받기
처음 한 번만 인터넷에서 받고, 이후엔 컴퓨터에 저장된 것(`~/.keras/datasets`, `~/scikit_learn_data`)을 씁니다.

In [ ]:
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()
print("✅ MNIST", X_train.shape)
(X_train, y_train), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()
print("✅ Fashion MNIST", X_train.shape)
from sklearn.datasets import fetch_california_housing, load_sample_image
print("✅ California housing", fetch_california_housing().data.shape)
print("✅ 샘플 이미지", load_sample_image("china.jpg").shape)

In [ ]:
import os
print("현재 폴더:", os.getcwd())
print("✅ Salary2024.csv 있음" if os.path.exists("Salary2024.csv") else "⚠️ Salary2024.csv 없음 → Pandas 문제용 CSV를 이 노트북과 같은 폴더에 두세요")

## 속도 측정
수업 모델 기준으로 1 epoch에 몇 초 걸리는지 잽니다.

In [ ]:
import time
X = X_train / 255.0
dense = keras.Sequential([keras.layers.Input(shape=(28, 28)), keras.layers.Flatten(),
                          keras.layers.Dense(300, activation="relu"), keras.layers.Dense(100, activation="relu"),
                          keras.layers.Dense(10, activation="softmax")])
dense.compile(loss="sparse_categorical_crossentropy", optimizer="sgd", metrics=["accuracy"])
t = time.time(); dense.fit(X[:55000], y_train[:55000], epochs=1, verbose=0); t_dense = time.time() - t

cnn = keras.Sequential([keras.layers.Input(shape=(28, 28, 1)),
                        keras.layers.Conv2D(32, 3, padding="same"), keras.layers.MaxPooling2D(2),
                        keras.layers.Conv2D(64, 3, padding="same"), keras.layers.MaxPooling2D(2),
                        keras.layers.Flatten(), keras.layers.Dense(128, activation="relu"),
                        keras.layers.Dropout(0.2), keras.layers.Dense(10, activation="softmax")])
cnn.compile(loss="sparse_categorical_crossentropy", optimizer="nadam", metrics=["accuracy"])
t = time.time(); cnn.fit(X[:55000, ..., np.newaxis], y_train[:55000], epochs=1, verbose=0); t_cnn = time.time() - t

print(f"Dense 300-100 : 1 epoch {t_dense:.0f}초 → 10 epoch 약 {t_dense*10/60:.1f}분")
print(f"CNN (7-1 구조): 1 epoch {t_cnn:.0f}초 → 10 epoch 약 {t_cnn*10/60:.1f}분")
print("✅ 점검 끝. 시험 때 시간이 빠듯하면 epochs를 줄이고 verbose=2로 실행하세요.")